# Документ OCR pipeline (Google Colab)

Рабочий baseline для тестового задания ML Engineer:
1. Загружает фото документа
2. Выравнивает документ (perspective transform)
3. Распознаёт текст (EasyOCR, локально, без токенов)
4. Извлекает структурированные поля (ФИО, дата рождения, номер документа)
5. Сохраняет выровненное изображение, изображение с боксами и JSON


In [ ]:
!pip -q install opencv-python-headless easyocr matplotlib rapidfuzz pillow

In [ ]:
import re
import json
import unicodedata
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr
import torch
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

In [ ]:
# --- Геометрия: выравнивание документа ---
def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype='float32')

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped


def align_document(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(gray, 60, 180)

    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    doc_cnt = None
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            doc_cnt = approx.reshape(4, 2)
            break

    if doc_cnt is None:
        # fallback: если контур не найден, возвращаем исходное изображение
        return image_bgr

    aligned = four_point_transform(image_bgr, doc_cnt.astype('float32'))
    return aligned

In [ ]:
# --- OCR + визуализация ---
reader = easyocr.Reader(['ru', 'en'], gpu=torch.cuda.is_available())


def run_ocr(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = reader.readtext(rgb, detail=1, paragraph=False)
    return results


def load_font(size=20):
    font_candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
    ]
    for font_path in font_candidates:
        try:
            return ImageFont.truetype(font_path, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


def draw_boxes(image_bgr, ocr_results):
    # PIL нужен для корректной отрисовки русского текста (без ?)
    rgb = cv2.cvtColor(image_bgr.copy(), cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(rgb)
    draw = ImageDraw.Draw(image_pil)
    font = load_font(size=max(14, image_bgr.shape[1] // 60))

    for item in ocr_results:
        box, text, conf = item
        pts = [(int(p[0]), int(p[1])) for p in box]
        draw.line(pts + [pts[0]], fill=(0, 255, 0), width=3)

        x, y = pts[0]
        label = f'{text[:40]} ({float(conf):.2f})'
        draw.text((x, max(0, y - 24)), label, fill=(255, 0, 0), font=font)

    out_rgb = np.array(image_pil)
    return cv2.cvtColor(out_rgb, cv2.COLOR_RGB2BGR)

In [ ]:
# --- Извлечение полей из OCR текста (приоритет: русский) ---
date_pattern = re.compile(r'\b(\d{2}[./-]\d{2}[./-]\d{4})\b')
doc_number_pattern = re.compile(r'\b(\d{2}\s?\d{2}\s?\d{6}|\d{9,12}|\d{10})\b')
fio_pattern = re.compile(r'^[А-ЯЁ][А-ЯЁ-]+(?:\s+[А-ЯЁ][А-ЯЁ-]+){1,2}$')

stop_phrases = {
    'ВОДИТЕЛЬСКОЕ УДОСТОВЕРЕНИЕ', 'УДОСТОВЕРЕНИЕ ЛИЧНОСТИ', 'ПАСПОРТ',
    'РОССИЙСКАЯ ФЕДЕРАЦИЯ', 'DRIVING LICENCE', 'DRIVER LICENSE', 'IDENTITY CARD',
}

blocked_geo_tokens = {
    'ОБЛ', 'ОБЛАСТЬ', 'ГОРОД', 'Г', 'МОСКВА', 'РАЙОН', 'РЕСПУБЛИКА', 'КРАЙ',
    'АВТ', 'ФЕДЕРАЦИЯ', 'РФ', 'RUS'
}

latin_to_cyr = str.maketrans({
    'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'H': 'Н', 'K': 'К', 'M': 'М',
    'O': 'О', 'P': 'Р', 'T': 'Т', 'X': 'Х', 'Y': 'У'
})


def clean_text(s):
    return re.sub(r'\s+', ' ', str(s).strip())


def normalize_ocr_text(s):
    s = clean_text(s).upper()
    s = unicodedata.normalize('NFKC', s)
    s = s.translate(latin_to_cyr)
    s = re.sub(r'[^А-ЯЁ\-\s0-9./]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def strip_to_name_chars(s):
    return re.sub(r'[^А-ЯЁ\-\s]', '', s).strip()


def name_word_score(word):
    if len(word) < 1:
        return 0
    if word in blocked_geo_tokens:
        return 0
    if re.fullmatch(r'[А-ЯЁ\-]+', word) is None:
        return 0
    # инициалы и короткие слова допускаем (иначе теряем 'Е Е')
    if len(word) == 1:
        return 1
    return 1 if len(word) == 2 else 2


def line_name_score(line):
    c = strip_to_name_chars(normalize_ocr_text(line))
    if not c or c in stop_phrases:
        return 0
    words = c.split()
    if not words:
        return 0
    if any(w in blocked_geo_tokens for w in words):
        return 0
    return sum(name_word_score(w) for w in words)


def likely_fio_line(line):
    c = strip_to_name_chars(normalize_ocr_text(line))
    if not c:
        return False
    if c in stop_phrases:
        return False
    words = c.split()
    if any(w in blocked_geo_tokens for w in words):
        return False
    return bool(fio_pattern.match(c))


def extract_line_items(ocr_results):
    items = []
    for box, text, conf in ocr_results:
        if float(conf) < 0.20:
            continue
        norm = normalize_ocr_text(text)
        if not norm:
            continue
        ys = [p[1] for p in box]
        xs = [p[0] for p in box]
        y_center = float(sum(ys) / len(ys))
        x_left = float(min(xs))
        items.append({
            'raw': str(text),
            'norm': norm,
            'conf': float(conf),
            'y_center': y_center,
            'x_left': x_left
        })
    items.sort(key=lambda d: (d['y_center'], d['x_left']))
    return items


def choose_full_name(line_items):
    lines = [x['norm'] for x in line_items]

    # 1) Прямое совпадение одной строкой из 2-3 слов
    for line in lines:
        if likely_fio_line(line):
            return strip_to_name_chars(line)

    # 2) Комбинация соседних строк (частый случай: Фамилия / Имя / Отчество)
    best = (0, None)
    n = len(lines)
    for i in range(n):
        for ln in (2, 3):
            if i + ln > n:
                continue
            chunk = lines[i:i+ln]
            merged = strip_to_name_chars(' '.join(chunk))
            words = merged.split()
            if not (2 <= len(words) <= 4):
                continue
            if any(w in blocked_geo_tokens for w in words):
                continue
            score = sum(line_name_score(x) for x in chunk)
            # штраф за слишком шумные строки
            if any(len(w) == 1 for w in words):
                score -= 1
            if score > best[0]:
                best = (score, merged)

    if best[1] and best[0] >= 3:
        return best[1]
    return None


def extract_fields(ocr_results):
    line_items = extract_line_items(ocr_results)
    lines = [x['norm'] for x in line_items]
    joined = ' '.join(lines)

    birth_date = None
    date_match = date_pattern.search(joined)
    if date_match:
        birth_date = date_match.group(1).replace('-', '.').replace('/', '.')

    doc_number = None
    normalized_joined = joined.replace('O', '0').replace('О', '0')
    num_match = doc_number_pattern.search(normalized_joined)
    if num_match:
        doc_number = re.sub(r'\s+', ' ', num_match.group(1)).strip()

    fio = choose_full_name(line_items)

    return {
        'full_name': fio,
        'birth_date': birth_date,
        'document_number': doc_number,
        'ocr_lines_normalized': lines
    }

In [ ]:
# --- Основной пайплайн ---
def to_python(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, list):
        return [to_python(x) for x in obj]
    if isinstance(obj, tuple):
        return [to_python(x) for x in obj]
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    return obj


def process_document_image(image_path, out_dir='outputs'):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f'Не удалось прочитать изображение: {image_path}')

    aligned = align_document(image_bgr)
    ocr_results = run_ocr(aligned)
    annotated = draw_boxes(aligned, ocr_results)
    fields = extract_fields(ocr_results)

    stem = Path(image_path).stem
    aligned_path = out_dir / f'{stem}_aligned.jpg'
    annotated_path = out_dir / f'{stem}_annotated.jpg'
    json_path = out_dir / f'{stem}_fields.json'

    cv2.imwrite(str(aligned_path), aligned)
    cv2.imwrite(str(annotated_path), annotated)

    payload = {
        'input_image': str(image_path),
        'aligned_image': str(aligned_path),
        'annotated_image': str(annotated_path),
        'fields': fields,
        'ocr': [
            {
                'box': to_python(item[0]),
                'text_raw': str(item[1]),
                'text_normalized': normalize_ocr_text(item[1]),
                'confidence': float(item[2])
            } for item in ocr_results
        ]
    }

    payload = to_python(payload)

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    # дополнительная версия с BOM для корректного открытия в некоторых редакторах Windows
    with open(str(json_path).replace('.json', '_utf8sig.json'), 'w', encoding='utf-8-sig') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return payload

In [ ]:
# --- Загрузка файла в Colab и запуск ---
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))

result = process_document_image(image_name, out_dir='outputs')
print(json.dumps(result['fields'], ensure_ascii=False, indent=2))

aligned_rgb = cv2.cvtColor(cv2.imread(result['aligned_image']), cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(cv2.imread(result['annotated_image']), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.title('Aligned')
plt.imshow(aligned_rgb)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Detections + OCR')
plt.imshow(annotated_rgb)
plt.axis('off')
plt.show()

In [ ]:
# --- Скачать артефакты на локальную машину ---
files.download(result['aligned_image'])
files.download(result['annotated_image'])
files.download(f"outputs/{Path(image_name).stem}_fields.json")